## Utility notebook: merge evaluation Excel reports

This archival notebook merges per-model evaluation workbooks into a single report workbook. It can delete and recreate the report folder, so double-check the resolved paths before running the final execution cell.

In [ ]:
from contextlib import suppress
from pathlib import Path
import shutil

import pandas as pd

# Advanced users: edit these paths directly in this notebook.
ROOT_PATH = Path("/home/vasakjakub/fenotypizace")
DRY_RUN = False
PROCESSED_EVALUATION_DIR = ROOT_PATH / "data" / "processed" / "evaluace_relearned_org"

print("Repo root:", ROOT_PATH)
print("Processed evaluation dir:", PROCESSED_EVALUATION_DIR)
print("DRY_RUN:", DRY_RUN)

2024-10-22 18:57:25.234163: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-22 18:57:25.234205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-22 18:57:25.271652: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-22 18:57:25.352385: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-22 18:57:26.278564: W tensorflow/compiler/tf2

In [ ]:
drive_path = ROOT_PATH
excel_fold = PROCESSED_EVALUATION_DIR

report_path = excel_fold / "report"
dest_path = report_path / "MERGE_report.xlsx"

print(report_path)

/home/james/projects/fenotypizace/source/data/processed/evaluace_relearned_org/report/


In [ ]:
def delete_and_create(report_path: Path) -> None:
    with suppress(FileNotFoundError):
        shutil.rmtree(report_path)
    report_path.mkdir(parents=True, exist_ok=True)


def excel_to_df(path: Path, sheet: str) -> tuple[pd.DataFrame, str]:
    name = path.stem
    df = pd.read_excel(path, sheet_name=sheet)
    df["Name"] = name
    columns = ["Name", "TP", "TN", "FP", "FN", "True", "False", "Accuracy"]
    return df[columns], name


def excel_list(excel_fold: Path) -> list[Path]:
    return sorted(
        [path for path in excel_fold.iterdir() if path.suffix in {".xls", ".xlsx"}]
    )


def eva_into_csvs(excel_fold: Path, report_path: Path) -> None:
    sheets = [
        "ground truth",
        "accuracy +-1",
        "accuracy +-2",
        "accuracy after clean",
        "accuracy after clean +-1",
        "accuracy after clean +-2",
    ]
    for path in excel_list(excel_fold):
        for sheet in sheets:
            df, _ = excel_to_df(path, sheet)
            file_path = report_path / f"{sheet}.csv"
            header = ["Name", "TP", "TN", "FP", "FN", "True", "False", "Accuracy"]

            if not file_path.is_file():
                df.to_csv(file_path, index=False, header=header)
            else:
                df.to_csv(file_path, mode="a", index=False, header=False)


def excel_merge(report_path: Path, dest_path: Path) -> None:
    csv_files = sorted(
        [path for path in report_path.iterdir() if path.suffix == ".csv"]
    )

    with pd.ExcelWriter(dest_path, engine="openpyxl") as excel_writer:
        for csv_file in csv_files:
            df = pd.read_csv(csv_file)
            df.to_excel(excel_writer, sheet_name=csv_file.stem, index=False)


def excel_to_df_box(
    path: Path,
    sheet: str,
    row_index: int,
) -> tuple[pd.DataFrame, str]:
    name = path.stem
    df = pd.read_excel(path, sheet_name=sheet)
    df["BOX_CONFUSION_MATRIX"] = name
    df = df.iloc[[row_index]]
    columns = ["BOX_CONFUSION_MATRIX", "TN", "TP", "FN", "FP", "acc"]
    return df[columns].rename(columns={"acc": "Accuracy"}), name


def eva_into_csvs_box(excel_fold: Path, report_path: Path) -> None:
    row_mapping = {
        0: "box_clean_prob",
        1: "box_clean_prob_+-1",
        2: "box_clean_prob_+-2",
    }
    for path in excel_list(excel_fold):
        for row_index, row_name in row_mapping.items():
            df, _ = excel_to_df_box(path, "BOX", row_index)
            file_path = report_path / f"{row_name}.csv"
            header = ["Name", "TN", "TP", "FN", "FP", "Accuracy"]

            if not file_path.is_file():
                df.to_csv(file_path, index=False, header=header)
            else:
                df.to_csv(file_path, mode="a", index=False, header=False)

In [ ]:
if DRY_RUN:
    print(
        "DRY_RUN is enabled. Review the resolved paths above, then set DRY_RUN = False to write files."
    )
else:
    delete_and_create(report_path)
    eva_into_csvs(excel_fold, report_path)
    eva_into_csvs_box(excel_fold, report_path)
    excel_merge(report_path, dest_path)

In [5]:
# shutil.rmtree("/content/Bucket/Fenotypizace/Data/Evaluation-old", ignore_errors=True)

In [6]:
# m_p = "/home/james/projects/fenotypizace/source/models/comparison/gdrive/ef_lstm_tuner_2-8_all_75_cropped75_org_nab_train"
# model = load_model(m_p)
# model.summary()